# Notebook 06 — Evaluación Final y Tablas para el Reporte IEEE

**Objetivo:** Compilar todos los resultados en tablas y figuras listas para el reporte académico.

**Prerequisito:** Notebooks 01-05 completados.

Contenido:
1. Tabla comparativa de todos los modelos
2. Tabla de IC bootstrap en test
3. Comparación con reducción de dimensión
4. Resumen del separability gap
5. Discusión automática de resultados

In [ ]:
# ════════════════════════════════════════════════════════════════
# SETUP — Colab sin auto-push (clone público con destino absoluto)
# ════════════════════════════════════════════════════════════════
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_DIR = f"/content/{REPO}"
    REPO_URL = "https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning.git"

    os.chdir("/content")

    for stray in (f"/{REPO}", REPO_DIR):
        if os.path.exists(stray) and not os.path.exists(f"{stray}/.git"):
            shutil.rmtree(stray, ignore_errors=True)

    if not os.path.exists(f"{REPO_DIR}/.git"):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")

    assert os.path.exists(f"{REPO_DIR}/.git"), f"git clone falló: {REPO_DIR}"

    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo local.")

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

In [ ]:
import sys
from pathlib import Path
_p = Path().resolve()
sys.path.insert(0, str(_p if (_p / 'src').exists() else _p.parent))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.io import load_json, load_config
%matplotlib inline
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Tabla comparativa de modelos clásicos

In [ ]:
metrics_dir = Path('artifacts/metrics')
model_names = ['logistic_regression', 'knn', 'random_forest', 'mlp', 'svm']

rows = []
for name in model_names:
    json_path = metrics_dir / f'{name}.json'
    if not json_path.exists():
        print(f'Falta {json_path} — ejecuta notebook 04')
        continue
    r = load_json(json_path)
    test = r['splits']['test']
    val  = r['splits']['val']
    ci   = test.get('bootstrap_ci_95', {})
    
    rows.append({
        'Modelo': name.replace('_', ' ').title(),
        'Val Acc':   round(val['accuracy'], 4),
        'Val F1':    round(val['f1_macro'], 4),
        'Test Acc':  round(test['accuracy'], 4),
        'Test F1':   round(test['f1_macro'], 4),
        'Test AUC':  round(test.get('roc_auc', float('nan')), 4),
        'Test BAcc': round(test['balanced_accuracy'], 4),
        'CI Acc 95%': f"[{ci.get('accuracy', {}).get('lower', float('nan')):.4f}, {ci.get('accuracy', {}).get('upper', float('nan')):.4f}]",
        'CI F1 95%':  f"[{ci.get('f1_macro', {}).get('lower', float('nan')):.4f}, {ci.get('f1_macro', {}).get('upper', float('nan')):.4f}]",
        'Train (s)': round(r.get('train_time_s', 0), 1),
    })

df_comp = pd.DataFrame(rows).sort_values('Test F1', ascending=False)
print('=== TABLA COMPARATIVA COMPLETA ===')
display(df_comp)
df_comp.to_csv(metrics_dir / 'final_comparison_table.csv', index=False)

# LaTeX para el reporte
print('\n=== LaTeX ===' )
print(df_comp[['Modelo', 'Test Acc', 'Test F1', 'Test AUC', 'Test BAcc', 'CI Acc 95%']].to_latex(index=False))

## 2. Figura resumen de todas las métricas

In [ ]:
metrics_to_plot = ['Test Acc', 'Test F1', 'Test AUC', 'Test BAcc']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(16, 5))
colors = plt.cm.tab10.colors

for ax, metric in zip(axes, metrics_to_plot):
    vals = df_comp[metric].values
    models = df_comp['Modelo'].values
    bars = ax.barh(models, vals, color=colors[:len(models)], alpha=0.8)
    ax.set_xlim(0.7, 1.01)
    ax.set_title(metric)
    ax.set_xlabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontsize=8)

fig.suptitle('Comparación de modelos — Embeddings contrastivos 1024D (Test)', fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/figures/final_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Impacto de la reducción de dimensión

In [ ]:
reeval_path = metrics_dir / 'reevaluation_reduction.json'
if reeval_path.exists():
    reeval = load_json(reeval_path)
    df_reeval = pd.DataFrame(reeval).T
    print('=== IMPACTO DE REDUCCIÓN DE DIMENSIÓN ===')
    display(df_reeval)
else:
    print('Ejecuta notebook 05 para generar este análisis')

## 4. Análisis de similitud

In [ ]:
sim_path = metrics_dir / 'similarity_summary.json'
if sim_path.exists():
    sim = load_json(sim_path)
    print('=== SIMILITUD COSENO ===')
    print(f"Intra-clase: {sim['intra_class']['mean']:.4f} ± {sim['intra_class']['std']:.4f}")
    print(f"Inter-clase: {sim['inter_class']['mean']:.4f} ± {sim['inter_class']['std']:.4f}")
    print(f"Separability gap: {sim['separability_gap']:.4f}")
    if sim['separability_gap'] > 0.1:
        print('✓ El encoder aprendió una representación discriminativa (gap > 0.1)')
    else:
        print('⚠ El gap es pequeño — considerar más épocas de entrenamiento')

## 5. Resumen ejecutivo

In [ ]:
# ════════════════════════════════════════════════════════════════
# DESCARGA MANUAL — baja el .ipynb ejecutado a tu PC
# Sin auto-push: tú subes/entregas el notebook por el medio que prefieras.
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "06_final_evaluation"
    # Forzar guardado del .ipynb (preserva outputs y figuras embebidas)
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    # Descargar a tu PC (revisa carpeta de descargas)
    from google.colab import files
    files.download(f"notebooks/{NOTEBOOK}.ipynb")
    print(f"✓ Descarga iniciada: {NOTEBOOK}.ipynb")

In [ ]:
if len(rows) > 0:
    best = df_comp.iloc[0]
    worst = df_comp.iloc[-1]
    print(f'\n=== RESUMEN EJECUTIVO ===')
    print(f'Mejor modelo:  {best["Modelo"]} — F1={best["Test F1"]:.4f} | AUC={best["Test AUC"]:.4f}')
    print(f'Peor modelo:   {worst["Modelo"]} — F1={worst["Test F1"]:.4f} | AUC={worst["Test AUC"]:.4f}')
    print(f'Spread en F1:  {best["Test F1"] - worst["Test F1"]:.4f}')
    print()
    print('Todos los modelos operan sobre embeddings de 1024D del encoder SupCon (ResNet18).')
    print('Split estratificado fijo seed=42. Dataset balanceado 50/50 (sin oversampling).')